# Restoration on Colab Free ($0)
Upload this notebook to Colab and select a **free GPU** runtime. Upload the prepared
`colab_restoration_v1.zip` to your private Drive folder `RestorationProject`.
No new dataset/weight downloads or package installs occur here. Data and training
stay on Colab; your Mac only displays the notebook. See the packaged cloud runbook.

This is a **new matched CUDA study**: rerun A/B/C together, never combine CUDA/FP16
A/C with the old MPS/FP32 B. Research novelty is not established. Setup, short tests
and a bounded profile precede an explicit long-run gate; **Run all does not start
long training by default**. Free GPU availability and session duration vary.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, stat, subprocess, sys, zipfile
from pathlib import PurePosixPath
from google.colab import drive
# This explicit cell requests access to your Drive in the cloud runtime.
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/RestorationProject')
bundle = drive_root / 'colab_restoration_v1.zip'
project = Path('/content/Image_Degradatation_Restoration')
EXPECTED_BUNDLE_SHA256 = '37066216dcce9ae8f53fa449b22dcdef20a7a83e098001bb39bd62baaf87cb3f'
assert bundle.is_file(), f'Upload the prepared ZIP to {bundle}'


## Verify and unpack into local VM storage
This refuses an existing project folder. On a new VM, use the same original ZIP.
Keep any prior VM artifacts before choosing a different folder; do not delete runs.


In [ ]:
digest = hashlib.sha256()
with bundle.open('rb') as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_BUNDLE_SHA256, 'Transfer SHA256 mismatch'
assert not project.exists(), 'Project already exists; keep it and skip extraction, or use a fresh VM'
with zipfile.ZipFile(bundle) as archive:
    names = set()
    for entry in archive.infolist():
        path = PurePosixPath(entry.filename)
        assert (path.parts and path.parts[0] == project.name and not path.is_absolute()
                and '..' not in path.parts and '\\' not in entry.filename
                and entry.filename not in names
                and not stat.S_ISLNK(entry.external_attr >> 16)), 'Unsafe ZIP member'
        names.add(entry.filename)
    assert archive.testzip() is None, 'ZIP CRC failure'
    archive.extractall('/content')
os.chdir(project)
(project / "reports").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(project))
sys.path.insert(0, str(project / 'notebooks'))
from cloud_workflow import verify_bundle, prepare_plan, seal_profile, run_arm, sha256
inventory = verify_bundle(project)
print('Verified image counts:', inventory['split_counts'])
print('Image MiB:', round(inventory['image_bytes'] / 1024**2, 2))


In [ ]:
import torch, numpy, PIL, yaml
from utils.runtime import environment, select_device
from utils.degradation import codec_metadata
device = select_device('cuda')  # Fail instead of accidentally training on CPU.
print(json.dumps({**environment(device), 'codec': codec_metadata()}, indent=2))
print('Total GPU GiB:', torch.cuda.get_device_properties(device).total_memory / 1024**3)
assert (2, 4) <= tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (3, 0)
# Keep the existing supported CUDA environment. Do not install the Mac lock here.
# Missing/incompatible dependencies are a visible failure for review, not an auto-install.


## Short QA; no research training
The CPU fixtures verify shared mechanics and snapshot recovery. The two-update CUDA
smoke test checks basic device operation; the following full-model FP16 profile is
still necessary. Test output is not a scientific result.


In [ ]:
def checked(*args):
    command = [sys.executable, *map(str, args)]
    with subprocess.Popen(command, cwd=project, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as child:
        lines = []
        for line in child.stdout:
            print(line, end='', flush=True)
            lines.append(line)
        code = child.wait()
    output = ''.join(lines)
    if code:
        raise subprocess.CalledProcessError(code, command, output=output)
    return subprocess.CompletedProcess(command, code, stdout=output)
(project / 'reports').mkdir(parents=True, exist_ok=True)
checked('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_*.py', '-v')
checked('notebooks/test_cloud_workflow.py')
checked('tests/smoke_test.py')
checked('train.py', '--dry-run', '--epochs', '1', '--batch-size', '2', '--device', 'cuda')


## Plan only, then profile the real workload
All three arms: seed 42, 5,000 updates, 128x128, clean microbatch 2, accumulation 4,
CUDA FP16, same LR schedule and two-view budget. Provisional cap: **3,600 seconds
per invocation, 10,800 seconds combined**, plus setup/final I/O. This is a proposal,
not approval. Do not change only one arm's microbatch/precision/horizon after starting.


In [ ]:
STUDY_NAME = 'cuda_jpeg_5000_v1'  # Use a new name for a genuinely new study.
plan_dir = project / 'reports' / STUDY_NAME
plan_file = prepare_plan(project, plan_dir, max_seconds=3600)
print(plan_file.read_text())


In [ ]:
profile_file = plan_dir / 'cuda_profile.json'
checked('scripts/profile.py', '--config', plan_dir / 'C.yaml', '--device', 'cuda',
        '--warmup', '10', '--steps', '50', '--max-seconds', '300', '--output', profile_file)
reviewed_plan = seal_profile(project, plan_file, profile_file)
profile = json.loads(profile_file.read_text())
profile_summary = json.loads((Path(profile['run_dir']) / 'summary.json').read_text())
print('Profile skipped updates:', profile_summary['skipped_updates'])
print(reviewed_plan.read_text())
# Save the reviewed plan and profile before any long training. Keep existing files.
study_backup = drive_root / 'studies' / STUDY_NAME
study_backup.mkdir(parents=True, exist_ok=True)
for source in (reviewed_plan, profile_file):
    target = study_backup / source.name
    if target.exists():
        assert sha256(target) == sha256(source), 'Existing study differs; retain it and choose a new name'
    else:
        shutil.copyfile(source, target)
print('Review the budget plus validation/backup overhead; the update projection excludes these.')


## Long-run approval gate
Leave the flag **False** until the actual profile and bounded three-arm budget have
been reviewed and explicitly approved. The default stops before research. Setting
it True and executing the next cells deliberately launches the reviewed study.
No paid resources, downloads, protected-test scoring or automatic expansion.

Each arm uses local VM storage and saves verified snapshots to Drive after the
first 100-update checkpoint, roughly every 200 updates, and on normal/error exit.
Keep roughly 2–3 GB Drive headroom. Abrupt termination can lose work since the last
finished snapshot. A failed/incomplete arm must stop the study before the next arm.


In [ ]:
APPROVED_LONG_RUN = False
assert APPROVED_LONG_RUN, 'Stop: inspect the actual CUDA profile and approve the bounded plan first'
results = {}


In [ ]:
assert APPROVED_LONG_RUN
results['A'] = run_arm(project, reviewed_plan, 'A_seed42', drive_root / 'studies',
                            approved=APPROVED_LONG_RUN)
print(results['A'])


In [ ]:
assert APPROVED_LONG_RUN
assert results.get('A', {}).get('completed') is True, 'Complete/recover A before B'
results['B'] = run_arm(project, reviewed_plan, 'B_seed42', drive_root / 'studies',
                            approved=APPROVED_LONG_RUN)
print(results['B'])


In [ ]:
assert APPROVED_LONG_RUN
assert all(results.get(arm, {}).get('completed') is True for arm in ('A', 'B')), 'Complete/recover A and B before C'
results['C'] = run_arm(project, reviewed_plan, 'C_seed42', drive_root / 'studies',
                            approved=APPROVED_LONG_RUN)
print(results['C'])


## Verify development checkpoints and save reports
After all arms complete, replay selected best checkpoints on the same dev8 panel.
These are development results, not protected tests. Inspect counts/skips and
compare only matched arms; do not infer publishable novelty from a positive score.


In [ ]:
assert APPROVED_LONG_RUN and set(results) == {'A', 'B', 'C'}
for arm in 'ABC':
    checkpoint = Path(results[arm]['run_dir']) / 'best_model.pth'
    output = plan_dir / f'{arm}_selected_dev.json'
    checked('eval.py', '--checkpoint', checkpoint, '--device', 'cuda', '--output', output)
    target = study_backup / output.name
    assert not target.exists(), 'Retain the existing report; choose a new report name'
    shutil.copyfile(output, target)
print('Saved selected development reports and checkpoint snapshots:', study_backup)


## Recovery only (skip new-study/training cells above on a replacement VM)
After setup and integrity checks, set the original study name and a complete private
snapshot ZIP. Restore the original reviewed plan/profile. The helper refuses a
changed GPU/software/codec, mismatched source or incompatible schedule. Review
remaining wall-time and **separately approve this resumed invocation**. It does not
automatically resume or restart other arms. Retain all original snapshots.

After recovery, execute only the remaining arms with `run_arm` and the original
reviewed plan, after checking saved completed-arm summaries. Skip the fresh-study
A/B/C cells above; their guards intentionally require verified prior results.


In [ ]:
RESUME_SNAPSHOT = None  # Example: Path('/content/drive/MyDrive/.../step_000100_micro_00000400.zip')
RESUME_APPROVED = False
INTERRUPTED_ARM = 'A_seed42'  # Set the actual interrupted arm; never restart a completed arm.
if RESUME_SNAPSHOT is not None:
    assert RESUME_APPROVED, 'Review remaining budget before resuming this invocation'
    original_study = drive_root / 'studies' / 'cuda_jpeg_5000_v1'
    original_plan = original_study / 'reviewed_plan.json'
    original = json.loads(original_plan.read_text())
    local_profile = project / original['cuda_profile']['path']
    local_profile.parent.mkdir(parents=True, exist_ok=True)
    if local_profile.exists():
        assert sha256(local_profile) == original['cuda_profile']['sha256']
    else:
        shutil.copyfile(original_study / 'cuda_profile.json', local_profile)
    resumed = run_arm(project, original_plan, INTERRUPTED_ARM, drive_root / 'studies',
                      approved=RESUME_APPROVED, resume_archive=Path(RESUME_SNAPSHOT))
    print(resumed)


Colab Free has variable GPU availability and runtime limits:
[official FAQ](https://research.google.com/colaboratory/faq.html).
No CUDA result is claimed until these cells have actually run on the cloud VM.
The Mac source fingerprints and completed local experiments are preserved.
